## 查看所有初始化参数

官方文档会列出通用参数和各 provider 的集成说明，但不同模型类支持的完整参数集合会随 provider、版本和底层 SDK 变化。

因此，除了查官方文档和 reference，也可以结合本地模型类的 `model_fields` 反查当前安装版本中实际暴露的参数。

以ChatDeepSeek类为例，其参数可以由自身定义或从父类BaseChatModel继承。直接查看源码也很难拼凑完整列表。这里通过查看ChatDeepSeek的类属性model_fields来获得完整参数列表。

In [1]:
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint
rprint(ChatDeepSeek.model_fields.keys())


dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 
'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 
'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 
'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 
'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning',
'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 
'http_socket_options', 'stream_chunk_timeout', 'stop', 'extra_body', 'include_response_headers', 'disabled_params',
'context_management', 'include', 'service_tier', 'store', 'truncation', 'use_previous_response_id', 
'use_responses_api', 'api_key', 'api_base'])

In [2]:
from langchain_openai import ChatOpenAI
from rich import print as rprint
rprint(ChatOpenAI.model_fields.keys())


dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 
'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 
'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 
'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 
'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning',
'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 
'http_socket_options', 'stream_chunk_timeout', 'stop', 'extra_body', 'include_response_headers', 'disabled_params',
'context_management', 'include', 'service_tier', 'store', 'truncation', 'use_previous_response_id', 
'use_responses_api'])

In [ ]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from rich import print as rich_print

load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    model="gpt-5.4-mini",
)
rich_print(type(model).model_fields.keys())  # Pydantic V2 中不推荐从实例访问 model_fields，推荐从模型类访问

In [3]:
from langchain_deepseek import ChatDeepSeek
for name, field in ChatDeepSeek.model_fields.items():
    print(name)
    print("  annotation:", field.annotation)
    print("  default:", field.default)
    print("  description:", getattr(field, "description", None))
    print("  alias:", field.alias)

name
  annotation: str | None
  default: None
  description: None
  alias: None
cache
  annotation: langchain_core.caches.BaseCache | bool | None
  default: None
  description: None
  alias: None
verbose
  annotation: <class 'bool'>
  default: PydanticUndefined
  description: None
  alias: None
callbacks
  annotation: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None
  default: None
  description: None
  alias: None
tags
  annotation: list[str] | None
  default: None
  description: None
  alias: None
metadata
  annotation: dict[str, typing.Any] | None
  default: None
  description: None
  alias: None
custom_get_token_ids
  annotation: collections.abc.Callable[[str], list[int]] | None
  default: None
  description: None
  alias: None
rate_limiter
  annotation: langchain_core.rate_limiters.BaseRateLimiter | None
  default: None
  description: None
  alias: None
disable_streaming
  annotation: typing.Union[bool, typing.Liter

## 模型类的参数构成

ChatDeepSeek 为例

**1、客户端与连接参数 (Networking)**

这类参数决定了代码“怎么连到服务端”，而不是“让模型怎么生成”。

| 参数名 | 说明 |
| ---- | ---- |
| api_key / openai_api_key | 鉴权密钥。DeepSeek 通常兼容 OpenAI 接口格式。 |
| api_base / openai_api_base | 接口地址（如 https://api.deepseek.com https://api.deepseek.com）。 |
| request_timeout / timeout | 网络请求超时时间。不同集成可能使用不同字段名或 alias，官方通用文档常写作 `timeout`。 |
| max_retries | 请求失败时的重试次数。 |
| http_client / http_async_client | 手动传入 httpx.Client 实例（用于更复杂的网络配置）。 |
| openai_proxy | 代理服务器配置。 |
| default_headers / default_query | 每次请求时默认携带的 HTTP Header 或 Query 参数。 |

**2、模型推理参数 (Model Inference)**
这些是直接传递给 DeepSeek 模型 API 的参数，决定了生成内容的质量和风格。

| 参数名 | 说明 |
| ---- | ---- |
| model_name | 指定具体的模型（如 deepseek-chat 或 deepseek-reasoning）。 |
| temperature | 采样温度，越高越随机。 |
| top_p | 核采样参数。 |
| max_tokens | 最大输出 token 数。 |
| stop | 停止符列表。 |
| streaming | 是否开启流式传输。 |
| n | 生成几个候选回复。 |
| reasoning | 是否启用推理模式 |
| reasoning_effort | (DeepSeek R1 特色) 控制思考链（COT）的深度。 |
| presence_penalty / frequency_penalty | 惩罚项(存在惩罚、频率惩罚)，用于减少内容重复。 |
| store | 是否存储对话。 |
| logit_bias | 调整特定词汇出现的概率。 |

**3、LangChain 框架通用参数**
由 LangChain 的 BaseChatModel 定义，所有其子类ChatXxx 都具备的，用于管理 LangChain 内部的逻辑（如日志、回调、元数据），仅在内部生效。

| 参数名 | 说明 |
| ---- | ---- |
| name | 给模型实例起个名字，用于在 Trace（如 LangSmith）中区分。 |
| verbose | 是否打印详细日志。 |
| callbacks | 回调处理器，用于集成 LangSmith 或自定义监控。 |
| tags / metadata | 用于标记该实例的标签和元数据。 |
| cache | 是否缓存该模型的请求结果。 |
| rate_limiter | LangChain 内部的频率限制器。 |


## model_kwargs 参数
用于存放底层 provider API 支持、但当前 LangChain wrapper 没有显式声明为字段的请求参数。

如用于支持Function Call的tools 字段。

说明：此处为了演示 `model_kwargs` 的透传作用，直接传递了工具调用接口。实际开发中不推荐采用这种原始方式，优先使用 LangChain 提供的 `bind_tools()` / tool calling 抽象。

实际开发中，可以对照 OpenAI Chat Completions [文档](https://platform.openai.com/docs/api-reference/chat-completions)或对应 provider 的官方 API 文档，确认底层请求支持哪些字段。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash", # 这里换成GPT第一次输出为空，因为其直接发起一个tool调用，没有额外解释，deepseek则有额外解释，所以content不为空
    model_kwargs={"tools": [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather of a location, the user should supply a location first.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. SanFrancisco, CA",
                        }
                    },
                    "required": ["location"]
                },
            }
        },
    ]}
)
# 向模型发送单条数据
response = model.invoke("你好，今天北京的天气如何")
# 打印响应
rprint(response)

AIMessage(
    content='你好！我来帮你查一下今天北京的天气情况。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道北京今天的天气。让我调用get_weather工具来获取北京的天气信息。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 75,
            'prompt_tokens': 304,
            'total_tokens': 379,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 18,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 304
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': 'f5b50d19-9d7d-45e2-be8d-a559c2c8df5a',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f185e-6e4c-7a21-8131-450660c74eef-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'location': 'Beijing, China'},
            'id': 'call_00_nyeBpcCPSjTr1vtEBlZH2232',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 304,
        'output_tokens': 75,
        'total_tokens': 379,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 18}
    }
)

## extra_body参数

`extra_body` 通常用于存放第三方 OpenAI-compatible provider 在标准 OpenAI API 协议之外扩展的非标准字段。

可以粗略理解为：`model_kwargs` 更像是补充传递标准请求参数；`extra_body` 更像是额外塞进 request body 的 provider-specific 扩展字段。

另外需要注意：如果用 `ChatOpenAI + base_url` 连接第三方 OpenAI-compatible 服务，LangChain 的 `ChatOpenAI` 主要按 OpenAI 官方规范解析响应，第三方非标准响应字段不一定会被提取或保留。需要深度使用 OpenRouter、DeepSeek 等厂商扩展能力时，优先考虑 provider-specific 集成包。

查阅OpenAI Chat Completions[文档](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)和[DeepSeek对话补全API文档](https://api-docs.deepseek.com/zh-cn/api/create-chat-completion)可知，thinking字段，用于控制是否启用思考模式。


In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
model="deepseek:deepseek-v4-flash",
extra_body={"thinking": {"type": "enabled"}},
)
# 向模型发送单条数据
response = model.invoke("一句话解释量子")
# 打印响应
rprint(response)

AIMessage(
    content='量子是物理量（如能量、角动量）的最小不可分割的离散单位。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们要求用一句话解释量子。量子是物理学中的一个基本概念，指物理量的最小不可分割的单位。比如光子是光的最小单位。但用
户只要求一句话解释，所以可以简洁地说：量子是物理量的最小离散单元。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 68,
            'prompt_tokens': 7,
            'total_tokens': 75,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 49,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 7
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '404496c7-d6cc-4dda-b2a6-9809014b5ce6',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f1862-6d38-7983-8185-4c05366aa0aa-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 7,
        'output_tokens': 68,
        'total_tokens': 75,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 49}
    }
)